<a href="https://colab.research.google.com/github/kimnguyen02/Module6_ITAI2372_AI_in_retail/blob/main/Module6_ITAI2372_AI_in_retail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install flask transformers torch pyngrok -q

### Writing the app

In [16]:
%%writefile app.py
from flask import Flask, request, render_template
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

app = Flask(__name__)

MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()


def generate_description(product_name, keywords):
    prompt = (
        "Task: Write a useful online store product description. "
        "Write exactly two complete sentences. "
        f"Product name: {product_name}. "
        f"Required keywords: {keywords}. "
        "Explain how the product may be used in everyday life. "
        "Write the actual description, not a summary of the inputs. "
        "Do not start with 'Product:' or 'Keywords:'. "
        "Do not invent features or specifications."
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            min_new_tokens=25,
            num_beams=5,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2
        )

    description = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

    return description


@app.route("/", methods=["GET", "POST"])
def home():
    product_name = ""
    keywords = ""
    description = None
    error = None

    if request.method == "POST":
        product_name = request.form.get("product_name", "").strip()
        keywords = request.form.get("keywords", "").strip()

        if not product_name or not keywords:
            error = "Please enter both a product name and keywords."
        else:
            try:
                description = generate_description(
                    product_name, keywords
                )
            except Exception:
                app.logger.exception("Generation failed")
                error = "Description generation failed. Please try again."

    return render_template(
        "index.html",
        product_name=product_name,
        keywords=keywords,
        description=description,
        error=error
    )


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)


Overwriting app.py


### Writing the template

In [22]:
%%writefile templates/index.html

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>AI Product Description Generator</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            max-width: 750px;
            margin: 40px auto;
            padding: 20px;
            background: #f4f6f8;
            color: #222;
        }
        .container {
            background: white;
            padding: 28px;
            border-radius: 12px;
        }
        input, textarea, button {
            width: 100%;
            box-sizing: border-box;
            padding: 12px;
            margin-top: 8px;
            margin-bottom: 18px;
            font-size: 16px;
        }
        button {
            background: #2457a7;
            color: white;
            border: none;
            border-radius: 6px;
            cursor: pointer;
        }
        .result {
            background: #eef5ff;
            padding: 18px;
            border-radius: 8px;
            white-space: pre-wrap;
            overflow-wrap: anywhere;
        }
        .error {
            color: #b00020;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>AI Product Description Generator</h1>
        <p>Create a product description using AI and your keywords.</p>

        <form method="POST">
            <label for="product_name">Product Name</label>
            <input
                type="text"
                id="product_name"
                name="product_name"
                placeholder="e.g., Stainless Steel Water Bottle"
                value="{{ product_name }}"
                required
            >

            <label for="keywords">Keywords (comma-separated)</label>
            <textarea
                id="keywords"
                name="keywords"
                rows="3"
                placeholder="e.g., reusable, durable, eco-friendly"
                required
            >{{ keywords }}</textarea>

            <button type="submit">Generate Description</button>
        </form>

        {% if description %}
            <h2>Generated Product Description</h2>
            <div class="result">{{ description }}</div>
        {% endif %}

        {% if error %}
            <p class="error">{{ error }}</p>
        {% endif %}
    </div>
</body>
</html>


Overwriting templates/index.html


In [19]:
!python app.py

Loading weights: 100% 282/282 [00:00<00:00, 4932.17it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
Press CTRL+C to quit
127.0.0.1 - - [29/Oct/2026 16:53:17] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [29/Oct/2026 16:54:03] "POST / HTTP/1.1" 200 -


Exception ignored in atexit callback <bound method TMonitor.exit of <TMonitor(Thread-1, stopped daemon 137456982427328)>>:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tqdm/_monitor.py", line 44, in exit
    self.join()
  File "/usr/lib/python3.13/threading.py", line 1095, in join
    self._handle.join(timeout)
KeyboardInterrupt: 


In [18]:
from pyngrok import ngrok

ngrok.kill()
public_url = ngrok.connect(5000)
print("Your app URL:", public_url)

Your app URL: NgrokTunnel: "https://stoneware-reversal-dominoes.ngrok-free.dev" -> "http://localhost:5000"
